# CSC 4110 Machine Learning: task notebook


| Field | Fill in |
|---|---|
| Task | 9.3 Publishing and Deployment |
| Owner (name) | Group G04 — All Five Members |
| Registration number | Group task — individual contributions recorded |
| Group | G04 |
| Dataset | Credit Risk Dataset |
| Date started | 26 November 2026 |


**Rules**
- Save this file in your group folder as `G<No>_U<task>_<short name>.ipynb`, exactly as shown under **Save as** in your group's paper, e.g. `G01_U2.1_cleaning.ipynb`.
- One Markdown heading per part, then its code cells. Comment every code cell.
- Use the exact variable and file names in **Graded evidence** of your task, and `random_state=42` everywhere.
- Reviewers add a cell headed **Review by <Name>** in the Reviews section. They never edit the owner's cells.

In [2]:
# Setup: set your group and task, connect Google Drive, and work inside the group folder
GROUP = "G04"        # your group, two digits
UNIT = "9.3"         # your task number: 2.1, 2.2, 3.1, 3.2, 4, 5, 6, 7, 8, 9.1, 9.2 or 9.3
RANDOM_STATE = 42    # use this everywhere so the grader gets the same numbers

import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted here:", e)   # normal when the grader runs this notebook

BASE = f"/content/drive/MyDrive/CSC4110_{GROUP}"
for sub in ["data", "notebooks", "models", "outputs", "app", "report", "tests"]:
    os.makedirs(f"{BASE}/{sub}", exist_ok=True)
os.chdir(BASE)   # every path below is relative to the group folder
print("Working in", os.getcwd())

Mounted at /content/drive
Working in /content/drive/MyDrive/CSC4110_G04


In [3]:
# Libraries used in most tasks, and a helper that saves every chart to outputs/
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def save_fig(name):
    """Save the current chart as outputs/<name>.png, then show it."""
    path = f"outputs/{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=120)
    print("Saved", path)
    plt.show()

In [ ]:

# Task 9.3: Verify project files before deployment

import os

# Check the current project location
print("CURRENT PROJECT FOLDER")
print(os.getcwd())

# Files needed for the final application
required_files = [
    "data/clean_credit.csv",
    "models/G04_model.joblib",
    "app/app.py",
    "app/requirements.txt",
    "report/model_card.md"
]

print("\nPROJECT FILE VERIFICATION")

for file in required_files:
    if os.path.isfile(file):
        print("FOUND:", file)
    else:
        print("MISSING:", file)

print("\nInitial project verification completed!")


CURRENT PROJECT FOLDER
/content/drive/MyDrive/CSC4110_G04

PROJECT FILE VERIFICATION
FOUND: data/clean_credit.csv
FOUND: models/G04_model.joblib
FOUND: app/app.py
FOUND: app/requirements.txt
FOUND: report/model_card.md

Initial project verification completed!



## Task 9.3(a): GitHub Project Documentation

This section documents the Mulika SACCO machine learning project, explains how to run the notebooks and application, and records the responsibilities of all five group members.


In [ ]:

# Task 9.3(a): Create the main project README

from pathlib import Path

readme_path = Path("README.md")

readme_content = """# Mulika SACCO — Loan Risk Screening

## Project Overview

This project was developed by Group G04 for CSC 4110
Machine Learning at Kibabii University.

It uses the Credit Risk Dataset to predict loan interest
rates and estimate the probability of loan default.

The prediction app supports loan officers and does not
automatically approve or reject loan applications.

## Dataset

Dataset: Credit Risk Dataset
Source: https://www.kaggle.com/datasets/laotse/credit-risk-dataset
Licence: CC0 — Public Domain

Original dataset: 32,581 records
Cleaned dataset: 32,409 records

## Machine Learning Models

The group implemented Linear Regression, Logistic
Regression, Decision Trees, Random Forest, Gradient
Boosting, KNN, SVM and Neural Networks.

The deployed demonstration uses a Random Forest
classification pipeline saved as:

models/G04_model.joblib

## Group Members and Responsibilities

| Member | Assigned Tasks |
|---|---|
| Festus Ligeyo | Task 2.1 and Task 7 |
| Collins Kipkorir | Task 2.2 and Task 8 |
| Chirchir Collins | Task 3.1 and Task 5 |
| Peter Mbugua | Task 3.2 and Task 9.1 |
| Hartie Omuruon | Task 4 and Task 6 |

Tasks 1, 9.2 and 9.3 involve the whole group.

## How to Run the Notebooks

1. Open the notebooks folder.
2. Open the required notebook in Google Colab.
3. Mount Google Drive and select the project folder.
4. Run the code cells in order.

## How to Run the Streamlit App

Install the required packages:

```bash
pip install -r app/requirements.txt
```

Start the application from the project root:

```bash
streamlit run app/app.py
```

The app loads the saved Random Forest pipeline and
collects borrower details to calculate default risk.

## Online Application

Public app URL: Not yet deployed.

This section will be updated after Streamlit Community
Cloud deployment.

## Guidance for SACCO Loan Officers

Enter the applicant's information into the screening
form and click Predict Default Risk.

The application estimates the probability of default
and indicates when additional loan-officer review is
required.

The 30% referral threshold is provisional and must be
validated before operational use.

The prediction supports human review and must never
be used as the sole reason to approve or reject a loan.

## Responsible AI

The model card is available at:

report/model_card.md

It documents model performance, fairness findings,
limitations, testing requirements and human oversight.

The current application is a coursework demonstration,
not an approved real-world lending system.
"""

# Avoid overwriting an existing README
if readme_path.exists():
    print("README.md already exists.")
    print("Existing file preserved for review.")
else:
    readme_path.write_text(readme_content, encoding="utf-8")
    print("README.md created successfully!")

print("README exists:", readme_path.is_file())


README.md created successfully!
README exists: True



## Task 9.3(b): Model Smoke Test

This section creates a smoke test that loads the saved Random Forest pipeline and predicts three records from the held-out test dataset.

The test stops with an error if model loading or prediction fails.


In [ ]:

%%writefile tests/smoke_test.py
# Group G04: Model smoke test

from pathlib import Path
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

# Locate the project root
ROOT = Path(__file__).resolve().parent.parent

# Locate the model and dataset
MODEL_PATH = ROOT / "models" / "G04_model.joblib"
DATA_PATH = ROOT / "data" / "clean_credit.csv"

# Check the required files
if not MODEL_PATH.is_file():
    raise FileNotFoundError("Trained model not found")

if not DATA_PATH.is_file():
    raise FileNotFoundError("Cleaned dataset not found")

# Load the saved model
model = joblib.load(MODEL_PATH)

# Load the cleaned dataset
df = pd.read_csv(DATA_PATH)

# Use the exact training features
features = [
    "person_age",
    "person_income",
    "person_home_ownership",
    "person_emp_length",
    "loan_intent",
    "loan_grade",
    "loan_amnt",
    "loan_percent_income",
    "cb_person_default_on_file",
    "cb_person_cred_hist_length"
]

X = df[features].copy()
y = df["loan_status"].astype(int)

# Reproduce previous-default encoding
X["cb_person_default_on_file"] = (
    X["cb_person_default_on_file"]
    .astype(str)
    .str.strip()
    .str.upper()
    .replace({"0": "N", "1": "Y"})
)

# Reproduce the original held-out test split
_, X_test, _, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Select three test records
test_rows = X_test.iloc[:3]

# Make predictions
predictions = model.predict(test_rows)
probabilities = model.predict_proba(test_rows)[:, 1]

# Verify the outputs
if len(predictions) != 3:
    raise RuntimeError("Incorrect number of predictions")

if not all(0 <= p <= 1 for p in probabilities):
    raise RuntimeError("Invalid default probabilities")

# Display the predictions
print("MULIKA SACCO MODEL SMOKE TEST")

for i, (prediction, probability) in enumerate(
    zip(predictions, probabilities), start=1
):
    print(
        f"Loan {i}: Predicted={prediction}, "
        f"Default probability={probability:.2%}"
    )

print("\nSMOKE TEST PASSED!")


Writing tests/smoke_test.py


In [ ]:

# Run the group model smoke test

!python tests/smoke_test.py


MULIKA SACCO MODEL SMOKE TEST
Loan 1: Predicted=0, Default probability=11.00%
Loan 2: Predicted=0, Default probability=20.00%
Loan 3: Predicted=0, Default probability=0.00%

SMOKE TEST PASSED!


In [ ]:

# Check the final publishing files

import os

for path in ["README.md", "tests/smoke_test.py"]:
    print(path, "FOUND" if os.path.isfile(path) else "MISSING")


README.md FOUND
tests/smoke_test.py FOUND


## Load the data

- Task 2.1 loads the original Kaggle file.
- Task 2.2 loads `data/stage1_<dataset>.csv`.
- Every other task loads `data/clean_<dataset>.csv`.

In [ ]:
# Load the data for this task (change the file name to your group's file)
DATA_FILE = "data/clean_<dataset>.csv"
df = pd.read_csv(DATA_FILE)
print(df.shape)
df.head()

## Part a) <title of the part, copied from the practical>

Write a python code to:
1. <step 1>
2. <step 2>

In [ ]:
# a1. <what this cell does>


## Part b) <title of the part, copied from the practical>

Write a python code to:
1. <step 1>
2. <step 2>

In [ ]:
# b1. <what this cell does>


## Part c) <title of the part, copied from the practical>

Write a python code to:
1. <step 1>
2. <step 2>

In [ ]:
# c1. <what this cell does>


## Written answer

## Task 9.3: Written Answer — Publishing and Deployment

To use the Mulika SACCO application, a user opens https://mulika-sacco-g04.streamlit.app/ in a web browser and enters the required loan application information, including an internally assigned loan grade.

After clicking Predict Default Risk, the application uses the saved Random Forest pipeline to estimate the probability of loan default and indicate whether additional loan-officer review is required.

One risk of publishing the application online is that users may enter sensitive financial information or rely too heavily on predictions that have not been fully validated.

To reduce these risks, the application should be used with fictional data during demonstrations, undergo security and fairness testing before real use, and always support rather than replace decisions made by qualified loan officers.



## Reviews

### Review by <Name>
A check, an improvement or a question.

## Evidence checklist (tick before you submit)

- [ ] Runtime → Restart session and run all: no errors
- [ ] The basic check below shows no missing names or files
- [ ] The written answer is complete and quotes my numbers
- [ ] The self-check cell from Moodle passes
- [ ] File → Save, and File → Save a copy in GitHub

In [5]:
# Basic check: are the required variables and files there? (names and files only;
# the self-check cell on Moodle also tests the values)
import glob
REQUIRED = {
    "2.1": (["df_raw", "df_stage1", "outlier_col", "lower", "upper"], ["data/stage1_*.csv"]),
    "2.2": (["df_clean", "X_train", "X_test", "y_train", "y_test", "scaler"],
            ["data/clean_*.csv", "outputs/hist_target.png", "outputs/box_feature.png",
             "outputs/bar_class.png", "outputs/scatter_best.png", "outputs/corr_heatmap.png"]),
    "3.1": (["lr_simple", "lr_multi", "metrics_multi", "b0_gd", "b1_gd", "cost_history"],
            ["outputs/simple_fit.png", "outputs/actual_vs_pred.png", "outputs/gd_cost.png"]),
    "3.2": (["poly_results", "ridge_results", "lasso_results", "results_reg"],
            ["outputs/under_over_fit.png", "outputs/regression_comparison.png"]),
    "4":   (["y_train", "y_test", "log_reg", "cm", "results_clf"], ["outputs/confusion_matrix.png"]),
    "5":   (["pipe", "cv_scores", "grid", "test_score", "threshold_results"],
            ["outputs/learning_curve.png", "outputs/validation_curve.png"]),
    "6":   (["tree3", "ig_root", "depth_results", "results_trees", "importances"],
            ["outputs/tree_depth3.png", "outputs/depth_curve.png", "outputs/feature_importance.png"]),
    "7":   (["inertias", "sil_scores", "best_k", "pca", "dbscan_labels", "Q", "success_rates"],
            ["outputs/elbow.png", "outputs/pca_clusters.png", "outputs/success_rate.png"]),
    "8":   (["w_and", "b_and", "mlp_results", "best_mlp", "activation_results", "mlp_reg_rmse"], ["outputs/loss_curve.png"]),
    "9.1": (["final_pipe"], ["models/*_model.joblib", "app/app.py", "app/requirements.txt",
                             "outputs/app_screenshot.png"]),
    "9.2": (["fairness_table", "flagged"], ["outputs/fairness.csv", "outputs/fairness.png",
                                            "report/model_card.md"]),
    "9.3": ([], ["tests/smoke_test.py", "README.md"]),
}
names, files = REQUIRED[UNIT]
missing_names = [n for n in names if n not in globals()]
missing_files = [f for f in files if not glob.glob(f)]
print(f"Task {UNIT}")
print("Missing variables:", missing_names or "none")
print("Missing files:", missing_files or "none")
print("PASS" if not (missing_names or missing_files) else "FIX the items above before you submit")

Task 9.3
Missing variables: none
Missing files: none
PASS


## Self-check

Paste the self-check cell for your task from its Moodle assignment below, then run it.

In [ ]:
# Paste the self-check cell from Moodle here
